# Python 数据分析项目实战规范

本篇承接课程中的 NumPy、Pandas、Matplotlib 和 RFM 综合案例，整理一套从原始数据到最终交付的项目实践方法。它不是新的网盘视频模块，而是对已学内容的工程化复盘：如何拆解问题、组织代码、检查数据、验证指标并输出可复用结果。

## 一、先定义问题，再选择工具

开始分析前，先写清楚四件事：

1. **分析对象**：订单、用户、商品还是时间序列；
2. **统计粒度**：一行代表一笔订单、一个用户还是一个用户年度；
3. **时间范围**：起止日期、自然年、滚动周期或固定截止日；
4. **交付目标**：表格、图表、数据库表，还是供后续模型使用的特征。

工具选择可以遵循以下顺序：

- NumPy：数组级数值计算；
- Pandas：表格读取、清洗、筛选、合并、分组；
- Matplotlib：静态探索图和分析报告图；
- Pyecharts：需要交互、旋转或缩放的图表；
- Excel/SQL：结果交付和下游使用。

## 二、推荐的项目目录

一个小型分析项目也应把原始数据、代码、结果和说明分开：

```text
analysis_project/
├── data/
│   ├── raw/          # 原始数据，只读保存
│   └── processed/    # 清洗后的中间数据
├── notebooks/        # 探索和说明性分析
├── src/              # 可复用的处理函数
├── reports/          # 图表、Excel 和分析报告
├── requirements.txt  # 依赖版本
└── README.md         # 数据口径、运行方式和结论
```

原始数据不要直接覆盖；生成文件使用明确的名称和日期，便于追溯同一项目的不同版本。

## 三、建立可复现的运行环境

课程使用的常见依赖包括 Pandas、NumPy、Matplotlib、SQLAlchemy、PyMySQL 和可选的 Pyecharts。

```text
pandas
numpy
matplotlib
openpyxl
sqlalchemy
pymysql
pyecharts
```

可以把依赖写入 `requirements.txt`，再在新的虚拟环境安装。版本是否固定取决于项目要求；交付项目建议记录实际版本。

```python
import sys
import pandas as pd
import numpy as np
import matplotlib

print(sys.version)
print("pandas", pd.__version__)
print("numpy", np.__version__)
print("matplotlib", matplotlib.__version__)
```

## 四、用函数拆分数据处理步骤

探索阶段可以逐段运行 Notebook，但稳定项目应把重复逻辑封装成函数。函数的输入、输出和关键规则要明确。

```python
import pandas as pd

def load_orders(path: str) -> pd.DataFrame:
    """读取订单数据并统一关键字段类型。"""
    orders = pd.read_excel(path)
    orders["提交日期"] = pd.to_datetime(
        orders["提交日期"],
        errors="coerce",
    )
    orders["订单金额"] = pd.to_numeric(
        orders["订单金额"],
        errors="coerce",
    )
    return orders

def clean_orders(orders: pd.DataFrame) -> pd.DataFrame:
    """删除关键字段缺失和业务确认的无效订单。"""
    result = orders.dropna(
        subset=["会员ID", "订单号", "提交日期", "订单金额"]
    ).copy()
    result = result[result["订单金额"] > 1]
    return result
```

清洗函数返回新对象，避免在多个分析步骤之间产生难以追踪的隐式修改。

## 五、数据质量检查模板

### 5.1 结构检查

```python
def profile_frame(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "unique": df.nunique(dropna=True),
    })

profile = profile_frame(df)
print(df.shape)
print(profile)
```

### 5.2 关键约束检查

```python
assert df["会员ID"].notna().all()
assert df["订单号"].notna().all()
assert (df["订单金额"] > 0).all()
assert df["提交日期"].notna().all()
```

如果业务允许空值或金额为零，应把断言改成对应业务规则，并在 README 中记录原因。

## 六、避免重复计算和数据泄漏

### 6.1 明确行粒度

订单表的一行是一笔订单，按用户聚合时应使用：

```python
user_stats = (
    orders.groupby("会员ID", as_index=False)
          .agg(
              order_count=("订单号", "nunique"),
              total_amount=("订单金额", "sum"),
          )
)
```

如果订单号可能重复，使用 `nunique()` 比 `count()` 更安全。

### 6.2 避免时间泄漏

计算某个时间点的用户特征时，只能使用该时间点之前的数据。RFM 中的截止日期就是一个时间边界；把未来订单混入历史特征，会让用户价值被高估。

```python
cutoff = pd.Timestamp("2018-12-31")
history = orders[orders["提交日期"] <= cutoff].copy()
```

## 七、指标计算与口径记录

每一个指标都应能回答“从哪张表、按什么粒度、用什么时间范围、采用什么聚合函数”。例如：

| 指标 | 口径示例 |
| --- | --- |
| 用户数 | `会员ID.nunique()` |
| 订单数 | `订单号.nunique()` |
| 销售额 | `订单金额.sum()` |
| 平均客单价 | 销售额 / 订单数 |
| 最近购买间隔 | 截止日期 - 用户最后订单日期 |

```python
summary = (
    history.groupby("会员ID", as_index=False)
           .agg(
               order_count=("订单号", "nunique"),
               total_amount=("订单金额", "sum"),
               last_order=("提交日期", "max"),
           )
)
summary["recency_days"] = (
    cutoff - summary["last_order"]
).dt.days
summary["avg_order_value"] = (
    summary["total_amount"] / summary["order_count"]
)
```

## 八、分组、分箱与标签设计

`groupby` 用于聚合，`pd.cut` 用于把连续数值划分为离散区间。分箱边界要覆盖数据范围，并写明边界规则。

```python
# 用分位数生成三个近似等量区间
q1, q2 = summary["total_amount"].quantile([0.33, 0.66])
amount_bins = [
    summary["total_amount"].min() - 1,
    q1,
    q2,
    summary["total_amount"].max(),
]
summary["amount_level"] = pd.cut(
    summary["total_amount"],
    bins=amount_bins,
    labels=["低", "中", "高"],
    include_lowest=True,
)
```

分位数适合探索性分组，但不一定符合业务阈值。正式项目要比较分位数方案和业务方案的分布，再选择可解释、可执行的规则。

## 九、分析结果的可视化检查

绘图既是展示工具，也是数据质量检查工具。

```python
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=100)

summary["total_amount"].plot(
    kind="hist", bins=30, ax=axes[0], title="销售额分布"
)
summary.groupby("amount_level")["会员ID"].count().plot(
    kind="bar", ax=axes[1], title="各金额等级用户数"
)

for ax in axes:
    ax.set_xlabel("数值")
    ax.grid(True, linestyle="--", alpha=0.3)

fig.tight_layout()
fig.savefig("reports/customer_summary.png")
plt.show()
```

如果图表出现极端长尾、类别缺失或数量异常，应回到清洗和指标定义步骤检查，而不是直接调整图表样式掩盖问题。

## 十、RFM 项目的运营交付

RFM 结果交付至少应包含三类内容：

1. **明细表**：会员 ID、年度、R/F/M 原始值、三个得分和组合编码；
2. **汇总表**：每个分组的用户数、占比、年度变化；
3. **解释表**：每个分组的用户特征、建议动作和优先级。

```python
group_report = (
    rfm.groupby(["year", "rfm_group"], as_index=False)
       .agg(
           user_count=("会员ID", "nunique"),
           avg_r=("r", "mean"),
           avg_f=("f", "mean"),
           avg_m=("m", "mean"),
       )
)
group_report["user_ratio"] = (
    group_report["user_count"]
    / group_report.groupby("year")["user_count"].transform("sum")
)
```

运营部门需要的是可执行的分组和策略，分析报告应把编码翻译成“谁、为什么、做什么”。

## 十一、结果校验与抽样复核

### 11.1 数量校验

```python
# 用户分组记录数
print(rfm.shape[0])

# 每年用户数量
print(rfm.groupby("year")["会员ID"].nunique())

# 每个分组的用户数
print(rfm.groupby("rfm_group")["会员ID"].nunique())
```

### 11.2 关键关系校验

```python
assert rfm["r"].ge(0).all()
assert rfm["f"].ge(1).all()
assert rfm["m"].gt(0).all()
assert rfm["rfm_group"].str.len().eq(3).all()
```

### 11.3 抽样复核

随机抽取几个用户，回查其原始订单，手工确认最近日期、订单次数、订单金额和最终分组。聚合代码通过并不代表业务口径一定正确。

## 十二、文件和数据库交付

```python
# Excel 交付
rfm.to_excel("reports/sales_rfm_score.xlsx", index=False)
group_report.to_excel(
    "reports/rfm_group_report.xlsx",
    index=False,
)

# SQL 交付示意
# rfm.to_sql(
#     "sales_rfm_score",
#     engine,
#     index=False,
#     if_exists="append",
# )
```

数据库连接信息不要写死在 Notebook 中，应该从环境变量或本地配置读取。导出时明确是否包含索引，避免生成 `Unnamed: 0` 一类的无意义字段。

## 十三、报告结构模板

一份易读的分析报告可以按以下顺序组织：

1. **背景与目标**：为什么分析，服务什么决策；
2. **数据说明**：来源、时间范围、字段、行粒度；
3. **清洗规则**：缺失值、重复值和异常值的处理；
4. **指标口径**：公式、聚合粒度和截止日期；
5. **核心结果**：表格、趋势图和分组图；
6. **用户或业务解释**：结果意味着什么；
7. **行动建议**：哪些群体优先做什么；
8. **限制条件**：样本偏差、阈值假设和数据缺口；
9. **附录**：代码版本、运行方式和字段字典。

结论要能回溯到数据和计算规则，避免只给出没有证据的判断。

## 十四、项目自查清单

- [ ] 原始数据已单独保存，分析过程不会覆盖原文件；
- [ ] 已明确行粒度、统计周期和截止时间；
- [ ] 日期、数值和分类字段类型正确；
- [ ] 缺失、重复和异常规则已记录；
- [ ] 订单数和用户数没有混淆；
- [ ] `concat` 与 `merge` 后行数符合预期；
- [ ] 分箱边界覆盖全部数据；
- [ ] R、F、M 的评分方向正确；
- [ ] 图表有标题、单位、图例和数据范围；
- [ ] 结果已导出并完成抽样复核；
- [ ] 数据库账号、密码和个人信息没有写入公开文件。

## 本篇小结

数据分析项目的质量来自完整闭环：问题定义决定口径，数据检查决定可信度，Pandas 操作决定计算过程，可视化帮助发现规律和错误，结果校验保证交付可靠，报告和数据表则让结论能够被业务使用。把这些步骤固定下来，课程中的单个 API 才能真正转化为稳定的项目能力。